# Langchain 을 이용해서 Naive RAG 구현해보기

- 우리나라의 소득세법을 가지고, 소득세에 관련된 질의응답을 할 수 있도록 만들어볼 예정이다.
- 이 notebook 파일에서는 [ollama](https://ollama.com/)를 가지고 진행한다.
- [ollama](https://ollama.com/)는 홈페이지의 가이드에 따라서 local 에 설치하면 된다.
- local 에서 ollama 로 사용할 LLM 모델은 `gpt-oss:20b` 이고, Embedding 모델은 `nomic-embed-text:v1.5` 이다.
- 아래의 명령어로 두 모델을 local 에 설치한 뒤 이 노트북을 실행해야한다.
    ```
    ollama pull gpt-oss:20b
    ollama pull nomic-embed-text:v1.5
    ```
- ollama 의 자세한 사용법은 [이곳](https://github.com/ollama/ollama/tree/main/docs)을 참고하면 된다.

- RAG 를 구현하기 위해서는 다음 pipeline 을 수행하게 된다.

1. 문서의 내용을 읽는다.
2. 문서를 나눈다.
   - 토큰 수 초과로 답변을 생성하지 못할 수 있고, 문서가 길면(input 이 길면) 답변 생성이 오래 걸린다.
3. 나눈 문서를 임베딩을 만들어 Vector DB 에 저장한다.
4. 질문이 들어오면, Vector DB 에서 질문과 관련된 문서를 검색을 한다.
5. 검색으로 가져온 문서를 LLM 에 질문과 함께 전달한다.

In [ ]:
from pathlib import Path


# 로컬에 저장할 모든 data 를 모아두는 디렉토리
DATA_DIR = Path("../data")

## 1. 문서의 내용을 읽는다.

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader


file_path = str(DATA_DIR / "tax_law.docx")  # 소득세법 문서
loader = Docx2txtLoader(file_path=file_path)
documents = loader.load()
len(documents)  # 전체 문서를 1개로

## 2. 문서를 나눈다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # 이 list 에 있는 seperator 들이 기본값이다
    chunk_size=1024,
    chunk_overlap=256,
)
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)  # 전체 문서를 여러 개로

## 3. 문서를 임베딩을 만들어 Vector DB 에 저장한다.

In [ ]:
from langchain_ollama import OllamaEmbeddings


embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")

In [ ]:
from langchain_chroma import Chroma


ALREADY_EMBEDDED = False  # 여기서는 로컬에 만들었는지 안 만들었는지를 간단하게 상수로 체크한다.

# 나눈 documents 를 embeddings 가지고 vector 로 만든다.
# 빈 Chroma 벡터 스토어 인스턴스 생성
persist_directory = str(DATA_DIR / "chroma_db")
collection_name = "income_tax_law_collection"
vector_store = Chroma(
    embedding_function=embeddings,  # OpenAI 임베딩 함수 (텍스트를 벡터로 변환)
    persist_directory=persist_directory,  # 벡터 데이터베이스를 디스크에 저장할 경로
    collection_name=collection_name,  # 컬렉션 이름 (데이터베이스 내 테이블 개념)
)

if not ALREADY_EMBEDDED:
    # 배치 처리 설정
    batch_size = 100  # 한 번에 처리할 문서 개수 (OpenAI API 토큰 한도 방지)

    # 전체 documents를 batch_size 개씩 나누어서 처리
    for i in range(0, len(documents), batch_size):  # 0부터 시작해서 batch_size씩 증가
        # 배치의 끝 인덱스 계산 (마지막 배치는 전체 문서 수를 초과하지 않도록)
        end = i + batch_size if i + batch_size < len(documents) else len(documents)

        # 현재 배치에 해당하는 문서들 추출 (슬라이싱)
        batched_document = documents[i:end]

        # 배치된 문서들을 벡터 스토어에 추가
        # 여기서 내부적으로 임베딩 생성 및 저장이 일어남
        vector_store.add_documents(batched_document)

        # 진행 상황 출력 (처리된 문서의 인덱스 범위)
        print(f"{i} 부터 {end - 1} 까지 처리")

    # 결과: 모든 문서가 배치 단위로 안전하게 벡터 데이터베이스에 저장됨
    # persist_directory 덕분에 데이터는 디스크에 저장되어 재사용 가능

## 4. 질문이 들어오면, Vector DB 에서 질문과 관련된 문서를 검색을 한다.

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = vector_store.search(query, search_type="similarity", k=3)
len(retrieved_docs)

In [ ]:
retrieved_docs = vector_store.search(query, search_type="mmr", k=5)
len(retrieved_docs)

### Vector Store 검색 방법 비교: Similarity vs MMR

#### 1. Similarity 검색 (유사도 검색)

- 순수한 의미적 유사도만을 기준으로 문서를 검색하는 방법입니다.
- 쿼리와 가장 유사한 벡터를 가진 문서들을 반환합니다.

##### 동작 방식

1. **쿼리 임베딩**: 검색 쿼리를 벡터로 변환
2. **유사도 계산**: 모든 문서 벡터와 쿼리 벡터 간의 코사인 유사도 계산
3. **정렬 및 반환**: 가장 높은 유사도 점수를 가진 k개 문서 반환

#### 2. MMR 검색 (Maximal Marginal Relevance)

- 쿼리와의 **관련성(Relevance)**과 결과 간의 **다양성(Diversity)**을 동시에 최적화하는 검색 방법입니다.
- 중복을 방지하면서 다양한 관점의 정보를 제공합니다.

##### 동작 방식

MMR은 다음 공식을 사용합니다.

```
MMR = λ × Sim(query, doc) - (1-λ) × max(Sim(doc, selected_docs))
```

**매개변수 설명**

- `λ` (lambda): 관련성과 다양성 간의 균형 조절 (0~1)
- `Sim(query, doc)`: 쿼리와 문서 간의 유사도
- `max(Sim(doc, selected_docs))`: 이미 선택된 문서들과의 최대 유사도

**단계별 처리**

1. 첫 번째 문서: 쿼리와 가장 유사한 문서 선택
2. 이후 문서들: 관련성이 높으면서 이미 선택된 문서들과는 다른 문서 선택
3. k개 문서 선택까지 반복

#### 3. 상세 비교 분석

##### 기본 특성 비교

| 구분           | Similarity 검색  | MMR 검색          |
|--------------|----------------|-----------------|
| **검색 기준**    | 순수 유사도만 고려     | 유사도 + 다양성 동시 고려 |
| **알고리즘 복잡도** | 단순 (O(n))      | 복잡 (O(k×n))     |
| **처리 속도**    | 빠름             | 상대적으로 느림        |
| **결과 특성**    | 높은 관련성, 낮은 다양성 | 중간 관련성, 높은 다양성  |
| **중복 방지**    | 없음             | 강력한 중복 방지       |

##### 성능 및 효과성 비교

| 평가 기준      | Similarity 검색 | MMR 검색 |
|------------|---------------|--------|
| **정확도**    | ⭐⭐⭐⭐⭐         | ⭐⭐⭐⭐   |
| **다양성**    | ⭐⭐            | ⭐⭐⭐⭐⭐  |
| **응답 속도**  | ⭐⭐⭐⭐⭐         | ⭐⭐⭐    |
| **포괄성**    | ⭐⭐            | ⭐⭐⭐⭐⭐  |
| **구현 난이도** | ⭐⭐            | ⭐⭐⭐⭐   |

##### 적용 시나리오 비교

| 사용 상황       | Similarity 검색 | MMR 검색        |
|-------------|---------------|---------------|
| **구체적 질문**  | ✅ 최적          | ⚠️ 과도할 수 있음   |
| **탐색적 질문**  | ⚠️ 제한적        | ✅ 최적          |
| **사실 확인**   | ✅ 최적          | ⚠️ 불필요한 정보 포함 |
| **종합적 분석**  | ❌ 부족          | ✅ 최적          |
| **실시간 응답**  | ✅ 최적          | ⚠️ 지연 가능      |
| **RAG 시스템** | ⚠️ 제한적        | ✅ 최적          |

#### 선택 가이드라인

##### Similarity 검색을 선택해야 하는 경우

- ✅ **명확하고 구체적인 답변**이 필요할 때
- ✅ **빠른 응답 속도**가 중요할 때
- ✅ **정확한 수치나 공식**을 찾을 때
- ✅ **간단한 사실 확인**이 목적일 때
- ✅ **리소스 제약**이 있는 환경

##### MMR 검색을 선택해야 하는 경우

- ✅ **복합적인 질문**에 대한 포괄적 답변이 필요할 때
- ✅ **다양한 관점의 정보**를 원할 때
- ✅ **RAG 시스템**에서 풍부한 컨텍스트가 필요할 때
- ✅ **탐색적 검색**을 수행할 때
- ✅ **정보의 품질과 다양성** 모두 중요할 때

##### 매개변수 튜닝

- **k값 조정**: similarity는 작게(1-3), MMR은 크게(5-10)
- **lambda 값**: MMR에서 0.7-0.8 권장 (관련성 70-80%, 다양성 20-30%)


## 5. 검색으로 가져온 문서를 LLM 에 질문과 함께 전달한다.

In [ ]:
from langchain_ollama import ChatOllama


llm = ChatOllama(model="gpt-oss:20b")

In [ ]:
prompt = f"""[Identity]
- 당신은 한국 최고의 세무사입니다.
- [Context] 를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question: {query}
"""
ai_message = llm.invoke(prompt)

In [ ]:
print("질문:", query)
print("LLM 답변:", ai_message.content)

### 랭체인에 있는 prompt 사용해서 질문해보기

In [ ]:
from langchain import hub


prompt = hub.pull("rlm/rag-prompt")
prompt

##### `as_retriever` 함수는 보통 다음의 파라미터를 받는다.
- `search_type`: 검색 방식
    - 일반적으로 "similarity", "similarity_score_threshold", "mmr"를 사용
- `search_kwargs`: 검색 방식별 옵션
    - 공통적으로 k, filter를 주로 쓰고, mmr일 때 fetch_k, lambda_mult, Pinecone일 때 namespace, index_name 등을 추가로 줄 수 있다.
    - 실제로 지원되는 키는 사용하는 VectorStore Wrapper 마다 다르므로 문서를 보는 것이 좋다.
    - Store 별 similarity_search / similarity_search_with_score / max_marginal_relevance_search 메서드 시그니처를 참고해 동일한 인자를 search_kwargs로 넘긴다고 생각하면된다.


In [ ]:
# QA Chain 을 만들어서 실행!

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# 이전 버전
# from langchain.chains import RetrievalQA
#
# qa_chain = RetrievalQA(
#     llm,
#     retriever=vector_store.as_retriever(),
#     chain_type_kwargs={"prompt": prompt}
# )
# qa_chain({"query": query})


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


qa_chain = (
    {
        "context": vector_store.as_retriever() | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

qa_chain.invoke(query)

### 위 chain 의 실행 flow chart

![flow chart of rag](../images/20250902/flow-chart-rag-example.png)